# NanoGPT Research

Use a GPU runtime. A100 or H100 is preferred; T4 also works but is slower.


## Setup

Run the setup cell first, then execute the experiment cells in order.


In [1]:
!pip install -q torch
import torch
import random
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch | CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# --- GPT model (inline) ---
class Head(torch.nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = torch.nn.Linear(n_embd, head_size, bias=False)
        self.query = torch.nn.Linear(n_embd, head_size, bias=False)
        self.value = torch.nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, n_embd, num_heads, head_size, block_size, dropout=0.0):
        super().__init__()
        self.heads = torch.nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = torch.nn.Linear(num_heads * head_size, n_embd)
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class Block(torch.nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)
        self.ffwd = torch.nn.Sequential(torch.nn.Linear(n_embd, 4*n_embd), torch.nn.ReLU(), torch.nn.Linear(4*n_embd, n_embd), torch.nn.Dropout(dropout))
        self.ln1 = torch.nn.LayerNorm(n_embd)
        self.ln2 = torch.nn.LayerNorm(n_embd)
    def forward(self, x):
        return x + self.ffwd(self.ln2(x + self.sa(self.ln1(x))))

class GPT(torch.nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = torch.nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = torch.nn.Embedding(block_size, n_embd)
        self.blocks = torch.nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = torch.nn.LayerNorm(n_embd)
        self.lm_head = torch.nn.Linear(n_embd, vocab_size, bias=True)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        dev = idx.device
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=dev))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        if targets is None: return logits, None
        return logits, F.cross_entropy(logits.view(B*T, -1), targets.view(B*T), ignore_index=-100)

print("GPT model defined.")

PyTorch | CUDA: True | Device: NVIDIA A100-SXM4-80GB
GPT model defined.


In [2]:
# (Optional) Verify device — run cell 2 first
import torch
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Device: NVIDIA A100-SXM4-80GB


## 2-Digit Addition


In [3]:
# 2-Digit: ab + cd = xyz (e.g. 45 + 67 = 112)
TRAIN_SIZE, TEST_SIZE = 2000, 2000
vocab_size, n_embd, n_head, n_layer, block_size = 12, 48, 2, 1, 9
lr, max_iters, eval_every = 1e-3, 25000, 2500
stoi = {str(i): i for i in range(10)}
stoi['+'], stoi['='] = 10, 11
itos = {v: k for k, v in stoi.items()}

def to_digits(n, pad=2):
    if n == 0: return [0] * pad
    digs = []
    while n: digs.append(n % 10); n //= 10
    digs.reverse()
    return [0] * (pad - len(digs)) + digs

def add_2digit(a, b):
    s = a + b
    return [(s // 100) % 10, (s // 10) % 10, s % 10]

def make_dataset_2d(examples):
    inputs, targets = [], []
    for (a, b) in examples:
        seq = to_digits(a, 2) + [stoi['+']] + to_digits(b, 2) + [stoi['=']] + add_2digit(a, b)
        inp, tgt = seq[:-1], seq[1:]
        masked = [-100] * len(tgt)
        for i in [5, 6, 7]: masked[i] = tgt[i]
        inputs.append(inp); targets.append(masked)
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

def evaluate_2d(model, examples):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for (a, b) in examples:
            ctx = to_digits(a, 2) + [stoi['+']] + to_digits(b, 2) + [stoi['=']]
            idx = torch.tensor([ctx], device=device)
            preds = []
            for _ in range(3):
                logits, _ = model(idx)
                next_tok = logits[0, -1, :].argmax().item()
                idx = torch.cat([idx, torch.tensor([[next_tok]], device=device)], dim=1)
                preds.append(next_tok)
            if preds == add_2digit(a, b): correct += 1
            total += 1
    model.train()
    return correct, total

torch.manual_seed(42); random.seed(42)
all_pairs = [(a, b) for a in range(100) for b in range(100)]
random.shuffle(all_pairs)
train_ex = all_pairs[:TRAIN_SIZE]; test_ex = all_pairs[TRAIN_SIZE:TRAIN_SIZE+TEST_SIZE]
print("=" * 60); print("  2-DIGIT ADDITION"); print("=" * 60)
print(f"  Train: {TRAIN_SIZE}, Test: {TEST_SIZE}")

model_2d = GPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
opt = torch.optim.AdamW(model_2d.parameters(), lr=lr)
tr_in, tr_tgt = make_dataset_2d(train_ex)
tr_in, tr_tgt = tr_in.to(device), tr_tgt.to(device)

for step in range(max_iters + 1):
    logits, loss = model_2d(tr_in, tr_tgt)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % eval_every == 0:
        tc, tt = evaluate_2d(model_2d, train_ex); ec, et = evaluate_2d(model_2d, test_ex)
        print(f"  step {step:5d} | loss {loss.item():.4f} | train {tc}/{tt} | TEST {ec}/{et}")

tc, tt = evaluate_2d(model_2d, train_ex); ec, et = evaluate_2d(model_2d, test_ex)
print("\n" + "=" * 60); print("  FINAL"); print("=" * 60)
print(f"  Seen: {tc}/{tt} ({100*tc/tt:.1f}%)  |  Unseen: {ec}/{et} ({100*ec/et:.1f}%)")

  2-DIGIT ADDITION
  Train: 2000, Test: 2000
  step     0 | loss 2.6974 | train 0/2000 | TEST 0/2000
  step  2500 | loss 0.2162 | train 1596/2000 | TEST 953/2000
  step  5000 | loss 0.2145 | train 1577/2000 | TEST 995/2000
  step  7500 | loss 0.2464 | train 1441/2000 | TEST 922/2000
  step 10000 | loss 0.0546 | train 1984/2000 | TEST 1308/2000
  step 12500 | loss 0.0519 | train 1992/2000 | TEST 1300/2000
  step 15000 | loss 0.0527 | train 1993/2000 | TEST 1275/2000
  step 17500 | loss 0.0377 | train 1994/2000 | TEST 1245/2000
  step 20000 | loss 0.2458 | train 1695/2000 | TEST 989/2000
  step 22500 | loss 0.0439 | train 1993/2000 | TEST 1135/2000
  step 25000 | loss 0.0860 | train 1920/2000 | TEST 1108/2000

  FINAL
  Seen: 1920/2000 (96.0%)  |  Unseen: 1108/2000 (55.4%)


## 3-Digit Addition


In [5]:
# 3-Digit: abc + def = ghij (e.g. 456 + 789 = 1245)
TRAIN_SIZE, TEST_SIZE = 5000, 2000
vocab_size, n_embd, n_head, n_layer, block_size = 12, 64, 2, 1, 12
lr, max_iters, eval_every = 1e-3, 30000, 3000
stoi = {str(i): i for i in range(10)}
stoi['+'], stoi['='] = 10, 11

def to_digits3(n, pad=3):
    if n == 0: return [0] * pad
    digs = []
    while n: digs.append(n % 10); n //= 10
    digs.reverse()
    return [0] * (pad - len(digs)) + digs

def add_3digit(a, b):
    s = a + b
    return [(s // 1000) % 10, (s // 100) % 10, (s // 10) % 10, s % 10]

def make_dataset_3d(examples):
    inputs, targets = [], []
    for (a, b) in examples:
        seq = to_digits3(a) + [stoi['+']] + to_digits3(b) + [stoi['=']] + add_3digit(a, b)
        inp, tgt = seq[:-1], seq[1:]
        masked = [-100] * len(tgt)
        for i in [7, 8, 9, 10]: masked[i] = tgt[i]
        inputs.append(inp); targets.append(masked)
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

def evaluate_3d(model, examples):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for (a, b) in examples:
            ctx = to_digits3(a) + [stoi['+']] + to_digits3(b) + [stoi['=']]
            idx = torch.tensor([ctx], device=device)
            preds = []
            for _ in range(4):
                logits, _ = model(idx)
                next_tok = logits[0, -1, :].argmax().item()
                idx = torch.cat([idx, torch.tensor([[next_tok]], device=device)], dim=1)
                preds.append(next_tok)
            if preds == add_3digit(a, b): correct += 1
            total += 1
    model.train()
    return correct, total

torch.manual_seed(42); random.seed(42)
all_pairs = [(a, b) for a in range(1000) for b in range(1000)]
random.shuffle(all_pairs)
train_ex = all_pairs[:TRAIN_SIZE]; test_ex = all_pairs[TRAIN_SIZE:TRAIN_SIZE+TEST_SIZE]
print("=" * 60); print("  3-DIGIT ADDITION"); print("=" * 60)
print(f"  Train: {TRAIN_SIZE}, Test: {TEST_SIZE}")

model_3d = GPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
opt = torch.optim.AdamW(model_3d.parameters(), lr=lr)
tr_in, tr_tgt = make_dataset_3d(train_ex)
tr_in, tr_tgt = tr_in.to(device), tr_tgt.to(device)

for step in range(max_iters + 1):
    logits, loss = model_3d(tr_in, tr_tgt)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if step % eval_every == 0:
        tc, tt = evaluate_3d(model_3d, train_ex); ec, et = evaluate_3d(model_3d, test_ex)
        print(f"  step {step:5d} | loss {loss.item():.4f} | train {tc}/{tt} | TEST {ec}/{et}")

tc, tt = evaluate_3d(model_3d, train_ex); ec, et = evaluate_3d(model_3d, test_ex)
print("\n" + "=" * 60); print("  FINAL"); print("=" * 60)
print(f"  Seen: {tc}/{tt} ({100*tc/tt:.1f}%)  |  Unseen: {ec}/{et} ({100*ec/et:.1f}%)")

  3-DIGIT ADDITION
  Train: 5000, Test: 2000
  step     0 | loss 2.8889 | train 0/5000 | TEST 0/2000
  step  3000 | loss 0.2190 | train 3543/5000 | TEST 1151/2000
  step  6000 | loss 0.1076 | train 4460/5000 | TEST 1426/2000
  step  9000 | loss 0.0819 | train 4655/5000 | TEST 1457/2000
  step 12000 | loss 0.0672 | train 4772/5000 | TEST 1471/2000
  step 15000 | loss 0.0590 | train 4822/5000 | TEST 1464/2000
  step 18000 | loss 0.0554 | train 4854/5000 | TEST 1459/2000
  step 21000 | loss 0.0479 | train 4911/5000 | TEST 1456/2000
  step 24000 | loss 0.1804 | train 4115/5000 | TEST 1209/2000
  step 27000 | loss 0.0730 | train 4814/5000 | TEST 1370/2000
  step 30000 | loss 0.0509 | train 4939/5000 | TEST 1415/2000

  FINAL
  Seen: 4939/5000 (98.8%)  |  Unseen: 1415/2000 (70.8%)


## Training Sweep

Runs the 2-digit and 3-digit experiments with training sizes 1000, 2000, and 5000. Set `FAST=True` for a shorter run.


In [7]:
# SWEEP: 2-digit (1k,2k,5k) + 3-digit (1k,2k,5k) — run cell 2 first
# Set FAST=True for ~2-3 min per run (fewer iters). False = full training.
FAST = True  # 8k/12k iters. False = 25k/30k iters.
import time

stoi = {str(i): i for i in range(10)}
stoi['+'], stoi['='] = 10, 11

def to_d2(n, pad=2):
    if n == 0: return [0] * pad
    d = []; x = n
    while x: d.append(x % 10); x //= 10
    d.reverse()
    return [0] * (pad - len(d)) + d

def add_2(a, b):
    s = a + b
    return [(s // 100) % 10, (s // 10) % 10, s % 10]

def to_d3(n, pad=3):
    if n == 0: return [0] * pad
    d = []; x = n
    while x: d.append(x % 10); x //= 10
    d.reverse()
    return [0] * (pad - len(d)) + d

def add_3(a, b):
    s = a + b
    return [(s // 1000) % 10, (s // 100) % 10, (s // 10) % 10, s % 10]

def run_2d(train_size, test_size=2000, max_iters=8000 if FAST else 25000, eval_every=2000 if FAST else 2500):
    vocab_size, n_embd, n_head, n_layer, block_size = 12, 48, 2, 1, 9
    torch.manual_seed(42); random.seed(42)
    pairs = [(a, b) for a in range(100) for b in range(100)]
    random.shuffle(pairs)
    tr, te = pairs[:train_size], pairs[train_size:train_size+test_size]
    def mk(ex):
        inp, tgt = [], []
        for (a,b) in ex:
            seq = to_d2(a) + [10] + to_d2(b) + [11] + add_2(a,b)
            m = [-100]*len(seq[1:])
            for i in [5,6,7]: m[i] = seq[i+1]
            inp.append(seq[:-1]); tgt.append(m)
        return torch.tensor(inp).to(device), torch.tensor(tgt).to(device)
    def ev(m, ex):
        m.eval()
        c = 0
        with torch.no_grad():
            for (a,b) in ex:
                ctx = to_d2(a) + [10] + to_d2(b) + [11]
                idx = torch.tensor([ctx], device=device)
                pred = []
                for _ in range(3):
                    lg, _ = m(idx)
                    nxt = lg[0,-1,:].argmax().item()
                    idx = torch.cat([idx, torch.tensor([[nxt]], device=device)], dim=1)
                    pred.append(nxt)
                if pred == add_2(a,b): c += 1
        m.train()
        return c
    tr_in, tr_tgt = mk(tr)
    m = GPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    for step in range(max_iters+1):
        _, loss = m(tr_in, tr_tgt)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if step % eval_every == 0:
            sc, st = ev(m, tr), ev(m, te)
            print(f"  2d train={train_size} step {step:5d} | loss {loss.item():.4f} | seen {sc}/{len(tr)} test {st}/{len(te)}")
    sc, st = ev(m, tr), ev(m, te)
    return 100*sc/len(tr), 100*st/len(te)

def run_3d(train_size, test_size=2000, max_iters=12000 if FAST else 30000, eval_every=3000 if FAST else 3000):
    vocab_size, n_embd, n_head, n_layer, block_size = 12, 64, 2, 1, 12
    torch.manual_seed(42); random.seed(42)
    pairs = [(a, b) for a in range(1000) for b in range(1000)]
    random.shuffle(pairs)
    tr, te = pairs[:train_size], pairs[train_size:train_size+test_size]
    def mk(ex):
        inp, tgt = [], []
        for (a,b) in ex:
            seq = to_d3(a) + [10] + to_d3(b) + [11] + add_3(a,b)
            m = [-100]*len(seq[1:])
            for i in [7,8,9,10]: m[i] = seq[i+1]
            inp.append(seq[:-1]); tgt.append(m)
        return torch.tensor(inp).to(device), torch.tensor(tgt).to(device)
    def ev(m, ex):
        m.eval()
        c = 0
        with torch.no_grad():
            for (a,b) in ex:
                ctx = to_d3(a) + [10] + to_d3(b) + [11]
                idx = torch.tensor([ctx], device=device)
                pred = []
                for _ in range(4):
                    lg, _ = m(idx)
                    nxt = lg[0,-1,:].argmax().item()
                    idx = torch.cat([idx, torch.tensor([[nxt]], device=device)], dim=1)
                    pred.append(nxt)
                if pred == add_3(a,b): c += 1
        m.train()
        return c
    tr_in, tr_tgt = mk(tr)
    m = GPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    for step in range(max_iters+1):
        _, loss = m(tr_in, tr_tgt)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if step % eval_every == 0:
            sc, st = ev(m, tr), ev(m, te)
            print(f"  3d train={train_size} step {step:5d} | loss {loss.item():.4f} | seen {sc}/{len(tr)} test {st}/{len(te)}")
    sc, st = ev(m, tr), ev(m, te)
    return 100*sc/len(tr), 100*st/len(te)

results = []
for ts in [1000, 2000, 5000]:
    t0 = time.time()
    s_seen, s_unseen = run_2d(ts)
    results.append(('2-digit', ts, s_seen, s_unseen, time.time()-t0))
    print(f"  -> 2-digit {ts} train: seen {s_seen:.1f}% unseen {s_unseen:.1f}% ({time.time()-t0:.1f}s)\n")
for ts in [1000, 2000, 5000]:
    t0 = time.time()
    s_seen, s_unseen = run_3d(ts)
    results.append(('3-digit', ts, s_seen, s_unseen, time.time()-t0))
    print(f"  -> 3-digit {ts} train: seen {s_seen:.1f}% unseen {s_unseen:.1f}% ({time.time()-t0:.1f}s)\n")

print("=" * 60)
print("  SWEEP SUMMARY")
print("=" * 60)
for task, tr, seen, un, sec in results:
    print(f"  {task} {tr} train: seen {seen:.1f}% unseen {un:.1f}% ({sec:.1f}s)")

  2d train=1000 step     0 | loss 2.7051 | seen 0/1000 test 0/2000
  2d train=1000 step  2000 | loss 0.0129 | seen 996/1000 test 352/2000
  2d train=1000 step  4000 | loss 0.0028 | seen 1000/1000 test 420/2000
  2d train=1000 step  6000 | loss 0.0005 | seen 1000/1000 test 431/2000
  2d train=1000 step  8000 | loss 0.0001 | seen 1000/1000 test 430/2000
  -> 2-digit 1000 train: seen 100.0% unseen 21.5% (106.2s)

  2d train=2000 step     0 | loss 2.6974 | seen 0/2000 test 0/2000
  2d train=2000 step  2000 | loss 0.2817 | seen 1473/2000 test 883/2000
  2d train=2000 step  4000 | loss 0.1453 | seen 1802/2000 test 1075/2000
  2d train=2000 step  6000 | loss 0.0905 | seen 1956/2000 test 1220/2000
  2d train=2000 step  8000 | loss 0.1089 | seen 1929/2000 test 1208/2000
  -> 2-digit 2000 train: seen 96.5% unseen 60.4% (128.8s)

  2d train=5000 step     0 | loss 2.6996 | seen 0/5000 test 0/2000
  2d train=5000 step  2000 | loss 0.0116 | seen 4998/5000 test 1984/2000
  2d train=5000 step  4000 | 